# 02 — 状態・入力・座標系

## 目的
次元が合うだけの誤接続を防ぐ。`x(24)`, `u(24)`, `rbdState(36)` は同じ24/36個の
「数」ではなく、順序・単位・frameを含む契約である。

実装対応:
- `legged_controllers/config/a1/task.info`
- `ocs2_legged_robot` の centroidal model helpers
- [`StateEstimateBase.cpp`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d/legged_estimation/src/StateEstimateBase.cpp)


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [1]:
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 主要ベクトル

\[
x=[v_{\mathrm{com}}(3),\,L/m(3),\,p_b(3),\,(\psi,\theta,\phi)(3),\,q_j(12)]
\]
\[
u=[f_c(12),\,v_j(12)]
\]

`rbdState(36)`:
ZYX(3), base position(3), joint angles(12), world angular velocity(3),
world linear velocity(3), joint velocities(12)。

- world = odom側、base = 胴体固定側
- 姿勢は ZYX の **格納順 yaw, pitch, roll**
- 関節順と接触脚順はコメント上で差がある箇所があるため、名前を正本にする


In [2]:
# 明示的なsliceで契約をコード化する。
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`x` を後続計算で使う明示的な中間量として設定する。 数式: `x = np.zeros(24)` の演算・変換をPythonで評価する。
x = np.zeros(24)
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`blocks_x` を後続計算で使う明示的な中間量として設定する。 数式: `blocks_x = {` の演算・変換をPythonで評価する。
blocks_x = {
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、直前の式・構造へ `"v_com": slice(0, 3), "L_over_m": slice(3, 6),` の要素または終端を対応付ける。
    "v_com": slice(0, 3), "L_over_m": slice(3, 6),
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、直前の式・構造へ `"base_position": slice(6, 9), "zyx": slice(9, 12),` の要素または終端を対応付ける。
    "base_position": slice(6, 9), "zyx": slice(9, 12),
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、直前の式・構造へ `"joint_angles": slice(12, 24),` の要素または終端を対応付ける。
    "joint_angles": slice(12, 24),
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、直前の式・構造へ `}` の要素または終端を対応付ける。
}
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`x[blocks_x["base_position"]]` を後続計算で使う明示的な中間量として設定する。 数式: `x[blocks_x["base_position"]] = [1.0, 2.0, 0.30]` の演算・変換をPythonで評価する。
x[blocks_x["base_position"]] = [1.0, 2.0, 0.30]
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`x[blocks_x["zyx"]]` を後続計算で使う明示的な中間量として設定する。 数式: `x[blocks_x["zyx"]] = [np.deg2rad(30), 0.0, 0.0]` の演算・変換をPythonで評価する。
x[blocks_x["zyx"]] = [np.deg2rad(30), 0.0, 0.0]
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`assert all(x[s].shape == (3,) for k, s in blocks_x.items() if k != "join…` を不変条件として即時検査する。 数式: `assert all(x[s].shape == (3,) for k, s in blocks_x.items() if k != "join…` の演算・変換をPythonで評価する。
assert all(x[s].shape == (3,) for k, s in blocks_x.items() if k != "joint_angles")
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`assert x[blocks_x["joint_angles"]].shape == (12,)` を不変条件として即時検査する。 数式: `assert x[blocks_x["joint_angles"]].shape == (12,)` の演算・変換をPythonで評価する。
assert x[blocks_x["joint_angles"]].shape == (12,)
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`x` をこの章の処理順に沿って実行する。
x


array([0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 1.    , 2.    ,
       0.3   , 0.5236, 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ,
       0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.    , 0.    ])

In [3]:
# body前方速度をworldへ回す。yaw=30 degならx/y双方に成分が出る。
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`Rz` の責務を独立関数として定義する。
def Rz(yaw):
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`c, s` を後続計算で使う明示的な中間量として設定する。 数式: `c, s = np.cos(yaw), np.sin(yaw)` の演算・変換をPythonで評価する。
    c, s = np.cos(yaw), np.sin(yaw)
    # 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])` の値を次の制御境界へ返す。 数式: `return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])` の演算・変換をPythonで評価する。
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`v_body` を後続計算で使う明示的な中間量として設定する。 数式: `v_body = np.array([0.5, 0.0, 0.0])` の演算・変換をPythonで評価する。
v_body = np.array([0.5, 0.0, 0.0])
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`yaw` を後続計算で使う明示的な中間量として設定する。 数式: `yaw = x[9]` の演算・変換をPythonで評価する。
yaw = x[9]
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`v_world` を後続計算で使う明示的な中間量として設定する。 数式: `v_world = Rz(yaw) @ v_body` の演算・変換をPythonで評価する。
v_world = Rz(yaw) @ v_body
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`print("v_body [m/s]:", v_body)` の観測値を表示して判定根拠を残す。 数式: `print("v_body [m/s]:", v_body)` の演算・変換をPythonで評価する。
print("v_body [m/s]:", v_body)
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`print("v_world [m/s]:", v_world)` の観測値を表示して判定根拠を残す。 数式: `print("v_world [m/s]:", v_world)` の演算・変換をPythonで評価する。
print("v_world [m/s]:", v_world)
# 背景: 同じ長さでもframe・単位・順序が異なる。目的: 状態・入力vectorの型契約を検査するため、`assert np.allclose(np.linalg.norm(v_world), np.linalg.norm(v_body))` を不変条件として即時検査する。 数式: `assert np.allclose(np.linalg.norm(v_world), np.linalg.norm(v_body))` の演算・変換をPythonで評価する。
assert np.allclose(np.linalg.norm(v_world), np.linalg.norm(v_body))


v_body [m/s]: [0.5 0.  0. ]
v_world [m/s]: [0.433 0.25  0.   ]


## よくある誤り
- `x[:3]`をbase位置だと思う（実際は正規化並進運動量/CoM速度）。
- `u[12:]`をトルクだと思う（実際は関節速度）。
- local IMU角速度とworld角速度を混ぜる。
- `LF,LH,RF,RH` と接触名の順を無検証で同じとみなす。

### 演習
すべての境界に `shape`, `unit`, `frame`, `leg order`, `rate` の5項目を書く。
これが書けない配列は、数式変更より先に調査する。


## 章固有の背景
                同じ長さのvectorでも、順序・frame・単位が違えば物理的には別の型である。

                ## 章固有の目的
                24状態、24入力、36 rigid-body stateのsliceを、変換symbolと数式に結び付ける。

                ## この章のASCIIデータフロー
                ```text
                rbdState(36: ZYX,p,q,omega,v,dq) -> CentroidalModelRbdConversions
                                            -> x(24: h/m,pose,q)
u(24: four world forces,dq*) -----------------> centroidal dynamics
                ```

                ## 上流C++ / faithful pseudocode と数式の行対応
                ```cpp
                // external/legged_control/legged_estimation/src/StateEstimateBase.cpp
rbdState.segment<3>(0) = zyx;       // [yaw,pitch,roll]
rbdState.segment<3>(3) = position;  // p_b^W
rbdState.segment<12>(6) = q;        // q_j
rbdState.segment<3>(18) = omegaW;   // omega_b^W
rbdState.segment<3>(21) = vW;       // v_b^W
rbdState.segment<12>(24) = dq;      // dq_j
// ocs2 centroidal helper contract from config/a1/task.info
x = [h_linear/m, h_angular/m, p_b, ZYX, q_j];
u = [f_LF^W,f_RF^W,f_LH^W,f_RH^W,dq_j];
                ```

                **事実のラベル**: `external/legged_control/` の記述はcommit
                `a7f381c0367e98e31c01336e678eef47e304d40d` の上流実装事実。数式展開はそのinterfaceを説明する理論。
                `src/legged_control_mujoco/` に言及した行はproject所有adapterの実装であり、
                ROS1/OCS2 SQP原実装とは同一ではない。

                ## 章固有の結論
                `x[:6]` は正規化centroidal momentum、`u[:12]` はworld contact force、`u[12:]` はjoint velocityであり、torqueはWBC後まで現れない。
